# Lake Polygon Filtering Notebook

This notebook does the following:

1. Reads a DEM GeoTIFF and computes slope in degrees.
2. Reads glacier outlines from a ZIP file.
3. Reads lake polygons from a shapefile.
4. Uses Google Earth Engine to calculate **mean NDSI** and **mean NIR** for each lake polygon.
5. Removes lake polygons with **NDSI > 0.4** and **NIR > 0.15**.
6. Removes polygons that **do not have a glacier within 10 km**.
7. Removes polygons with **mean slope > 15°**.
8. Saves the final result as a shapefile.

## Notes
- You must have a Google Earth Engine account enabled.
- Earth Engine Python authentication requires initialization with a **Cloud project ID**.
- Update all file paths and date range before running.
- This notebook assumes the lake and glacier data use valid polygon geometries.


In [ ]:
# Install required packages if needed
# Uncomment and run if your environment does not already have these packages.

# !pip install earthengine-api geemap geopandas rasterio rasterstats shapely fiona pyproj


In [1]:
import os
import json
import math
import zipfile
import tempfile
from pathlib import Path

import ee
import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import xy
from rasterstats import zonal_stats
from shapely.geometry import mapping


## 1) User inputs

Set your input paths, output folder, Earth Engine project ID, and imagery date range here.


In [2]:
# =========================
# USER INPUTS
# =========================

DEM_TIF = r"D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_inputs\output_hh.tif"
GLACIER_ZIP = r"D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_inputs\glims_download_19805.zip"
LAKE_SHP = r"D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_inputs\hkh_lakes_2020.zip"
OUTPUT_DIR = r"D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_outputs"
OUTPUT_SHP = os.path.join(OUTPUT_DIR, "filtered_lakes.shp")

# Earth Engine project ID (required for ee.Initialize)
EE_PROJECT_ID = "ee-msds24036"

# Date range for image composite used to compute NDSI and NIR
START_DATE = "2020-08-01"
END_DATE   = "2020-09-30"

# Thresholds
NDSI_THRESHOLD = 0.4
NIR_THRESHOLD = 0.15
GLACIER_DISTANCE_KM = 10
SLOPE_THRESHOLD_DEG = 15

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output shapefile will be saved to:")
print(OUTPUT_SHP)


Output shapefile will be saved to:
D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_outputs\filtered_lakes.shp


## 2) Authenticate and initialize Earth Engine

Run this cell once. If needed, it will prompt for authentication.


In [3]:
# Authenticate only once if needed, then initialize Earth Engine.
# If you are already authenticated on this machine, ee.Authenticate() may not ask again.

try:
    ee.Initialize(project=EE_PROJECT_ID)
    print("Earth Engine initialized successfully.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
    print("Earth Engine authenticated and initialized successfully.")


Earth Engine initialized successfully.


## 3) Helper functions

In [4]:
def compute_slope_from_dem(dem_tif):
    """
    Compute slope (degrees) from a DEM GeoTIFF using finite differences.
    Returns:
        slope_deg (2D numpy array)
        profile (raster profile)
    """
    with rasterio.open(dem_tif) as src:
        dem = src.read(1, masked=True).astype("float64")
        transform = src.transform
        profile = src.profile.copy()
        nodata = src.nodata

        # Pixel size
        xres = transform.a
        yres = abs(transform.e)

        dem_filled = dem.filled(np.nan)

        # Gradients along y (rows) and x (cols)
        dz_dy, dz_dx = np.gradient(dem_filled, yres, xres)

        slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
        slope_deg = np.degrees(slope_rad)

        # Keep nodata as nan
        slope_deg[np.isnan(dem_filled)] = np.nan

        profile.update(dtype="float32", count=1, nodata=np.nan)

    return slope_deg.astype("float32"), profile


def save_array_as_tif(array, reference_tif, out_tif):
    with rasterio.open(reference_tif) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", count=1, nodata=np.nan)

        with rasterio.open(out_tif, "w", **profile) as dst:
            dst.write(array.astype("float32"), 1)


def read_shapefile_from_zip(zip_path):
    """
    Extract the zip to a temp folder and read the first .shp found.
    """
    tmp_dir = r"D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_inputs" #tempfile.mkdtemp(prefix="glacier_zip_")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp_dir)

    shp_files = list(Path(tmp_dir).rglob("*.shp"))
    if not shp_files:
        raise FileNotFoundError("No .shp file found inside the glacier ZIP archive.")

    gdf = gpd.read_file(shp_files[0])
    return gdf, tmp_dir


def ensure_same_crs(gdf, target_crs):
    if gdf.crs is None:
        raise ValueError("Input GeoDataFrame has no CRS defined.")
    if gdf.crs != target_crs:
        return gdf.to_crs(target_crs)
    return gdf


def pick_utm_epsg_from_gdf(gdf):
    """
    Pick a UTM EPSG code based on the centroid of the dataset.
    Useful for distance and area calculations in meters.
    """
    centroid = gdf.to_crs(4326).unary_union.centroid
    lon, lat = centroid.x, centroid.y
    zone = int((lon + 180) / 6) + 1
    if lat >= 0:
        return f"EPSG:{32600 + zone}"
    else:
        return f"EPSG:{32700 + zone}"


def mask_s2_sr(image):
    """
    Simple Sentinel-2 SR cloud mask using QA60.
    Reflectance is scaled by 10000 in S2 SR Harmonized.
    """
    qa = image.select("QA60")
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
        qa.bitwiseAnd(cirrus_bit_mask).eq(0)
    )

    return (
        image.updateMask(mask)
             .divide(10000)
             .copyProperties(image, image.propertyNames())
    )


def geopandas_to_ee_featurecollection(gdf):
    """
    Convert a GeoDataFrame to ee.FeatureCollection.
    Handles simple JSON-serializable attributes.
    """
    features = []
    for _, row in gdf.iterrows():
        geom = ee.Geometry(mapping(row.geometry))
        props = {}
        for col in gdf.columns:
            if col == "geometry":
                continue
            value = row[col]
            if pd.isna(value):
                value = None
            elif isinstance(value, (np.generic,)):
                value = value.item()
            props[col] = value
        features.append(ee.Feature(geom, props))
    return ee.FeatureCollection(features)


def add_ndsi_nir_to_lakes_batched(lakes_gdf, start_date, end_date, batch_size=50, simplify_tolerance=0.0001):
    """
    Batched version to avoid Earth Engine request payload limits.
    """
    lakes_wgs84 = lakes_gdf.to_crs(4326).copy()

    if "lake_id" not in lakes_wgs84.columns:
        lakes_wgs84["lake_id"] = np.arange(1, len(lakes_wgs84) + 1)

    lakes_wgs84["geometry"] = lakes_wgs84.geometry.simplify(
        tolerance=simplify_tolerance,
        preserve_topology=True
    )

    all_stats = []

    for start in range(0, len(lakes_wgs84), batch_size):
        batch = lakes_wgs84.iloc[start:start + batch_size].copy()
        lakes_fc = geopandas_to_ee_featurecollection(batch)
        region = lakes_fc.geometry().bounds()

        s2 = (
            ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(start_date, end_date)
            .filterBounds(region)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
            .map(mask_s2_sr)
        )

        composite = s2.median()
        ndsi = composite.normalizedDifference(["B3", "B11"]).rename("NDSI")
        nir = composite.select("B8").rename("NIR")
        image = ndsi.addBands(nir)

        reduced = image.reduceRegions(
            collection=lakes_fc,
            reducer=ee.Reducer.mean(),
            scale=10
        )

        data = reduced.getInfo()

        for feat in data["features"]:
            props = feat["properties"]
            all_stats.append({
                "lake_id": props.get("lake_id"),
                "mean_ndsi": props.get("NDSI"),
                "mean_nir": props.get("NIR")
            })

        print(f"Processed batch {start} to {start + len(batch) - 1}")

    stats_df = pd.DataFrame(all_stats)
    result = lakes_wgs84.merge(stats_df, on="lake_id", how="left")
    return result


def add_glacier_distance_km(lakes_gdf, glaciers_gdf):
    metric_crs = pick_utm_epsg_from_gdf(lakes_gdf)

    lakes_m = lakes_gdf.to_crs(metric_crs).copy()
    glaciers_m = glaciers_gdf.to_crs(metric_crs).copy()

    # remove empty / missing geometries
    lakes_m = lakes_m[lakes_m.geometry.notna() & ~lakes_m.geometry.is_empty].copy()
    glaciers_m = glaciers_m[glaciers_m.geometry.notna() & ~glaciers_m.geometry.is_empty].copy()

    # repair invalid geometries
    lakes_m["geometry"] = lakes_m.geometry.make_valid()
    glaciers_m["geometry"] = glaciers_m.geometry.make_valid()

    # keep only polygonal glacier geometries after repair
    glaciers_m = glaciers_m[
        glaciers_m.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ].copy()

    glacier_union = glaciers_m.geometry.union_all()

    lakes_m["glacier_dist_km"] = lakes_m.geometry.apply(
        lambda geom: geom.distance(glacier_union) / 1000.0
    )

    return lakes_m


def add_mean_slope_to_polygons(polygons_gdf, slope_tif):
    with rasterio.open(slope_tif) as src:
        raster_crs = src.crs

    polygons_proj = polygons_gdf.to_crs(raster_crs).copy()

    stats = zonal_stats(
        vectors=polygons_proj,
        raster=slope_tif,
        stats=["mean"],
        nodata=np.nan,
        geojson_out=False
    )

    polygons_proj["mean_slope_deg"] = [s["mean"] for s in stats]
    return polygons_proj


## 4) Read inputs

In [5]:
# Read lake polygons
lakes = gpd.read_file(LAKE_SHP)
print("Lake polygons:", len(lakes))
print("Lake CRS:", lakes.crs)

# Read glacier polygons from ZIP
glaciers, glacier_tmp_dir = read_shapefile_from_zip(GLACIER_ZIP)
print("Glacier polygons:", len(glaciers))
print("Glacier CRS:", glaciers.crs)


Lake polygons: 8808
Lake CRS: EPSG:32643
Glacier polygons: 872552
Glacier CRS: EPSG:4326


## 5) Compute slope raster from DEM

In [7]:
#slope_array, slope_profile = compute_slope_from_dem(DEM_TIF)

SLOPE_TIF = os.path.join(OUTPUT_DIR, "dem_slope_deg.tif")
#save_array_as_tif(slope_array, DEM_TIF, SLOPE_TIF)

print("Slope raster saved to:")
print(SLOPE_TIF)


Slope raster saved to:
D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_outputs\dem_slope_deg.tif


## 6) Standardize CRS and clean geometries

In [6]:
# Reproject glacier polygons to lake CRS if needed
if lakes.crs is None:
    raise ValueError("Lake shapefile has no CRS defined.")
if glaciers.crs is None:
    raise ValueError("Glacier shapefile has no CRS defined.")

glaciers = ensure_same_crs(glaciers, lakes.crs)

# Fix invalid geometries if any
lakes = lakes[lakes.geometry.notnull()].copy()
glaciers = glaciers[glaciers.geometry.notnull()].copy()

lakes["geometry"] = lakes.buffer(0)
glaciers["geometry"] = glaciers.buffer(0)

lakes = lakes[~lakes.is_empty].copy()
glaciers = glaciers[~glaciers.is_empty].copy()


print("Cleaned lake polygons:", len(lakes))
print("Cleaned glacier polygons:", len(glaciers))


Cleaned lake polygons: 8808
Cleaned glacier polygons: 872552


## 7) Calculate NDSI and NIR for each lake polygon from Google Earth Engine

In [7]:
lakes_gee = add_ndsi_nir_to_lakes_batched(lakes, START_DATE, END_DATE)

print(lakes_gee[["lake_id", "mean_ndsi", "mean_nir"]].head())


Processed batch 0 to 49
Processed batch 50 to 99
Processed batch 100 to 149
Processed batch 150 to 199
Processed batch 200 to 249
Processed batch 250 to 299
Processed batch 300 to 349
Processed batch 350 to 399
Processed batch 400 to 449
Processed batch 450 to 499
Processed batch 500 to 549
Processed batch 550 to 599
Processed batch 600 to 649
Processed batch 650 to 699
Processed batch 700 to 749
Processed batch 750 to 799
Processed batch 800 to 849
Processed batch 850 to 899
Processed batch 900 to 949
Processed batch 950 to 999
Processed batch 1000 to 1049
Processed batch 1050 to 1099
Processed batch 1100 to 1149
Processed batch 1150 to 1199
Processed batch 1200 to 1249
Processed batch 1250 to 1299
Processed batch 1300 to 1349
Processed batch 1350 to 1399
Processed batch 1400 to 1449
Processed batch 1450 to 1499
Processed batch 1500 to 1549
Processed batch 1550 to 1599
Processed batch 1600 to 1649
Processed batch 1650 to 1699
Processed batch 1700 to 1749
Processed batch 1750 to 1799
P

In [8]:
lakes_gee = add_glacier_distance_km(lakes_gee, glaciers)

C:\Users\gg\AppData\Local\Temp\ipykernel_7724\3976558041.py:72: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = gdf.to_crs(4326).unary_union.centroid


In [10]:
lakes_gee.describe()

,OBJECTID_1,Id,OBJECTID,Shape_Leng,Shape_Le_1,Shape_Area,lake_id,mean_ndsi,mean_nir,glacier_dist_km
count,8808.000000,8808.000000,8808.000000,8808.000000,8808.000000,8.808000e+03,8808.000000,8773.000000,8773.000000,8808.000000
mean,4404.500000,1.816417,103.222298,113.406047,363.335395,1.548161e+04,4404.500000,0.004747,0.102598,0.848881
std,2542.794919,15.069553,287.696948,398.025070,559.302013,7.929145e+04,2542.794919,0.274575,0.073701,1.898890
min,1.000000,0.000000,0.000000,0.000000,12.032583,9.033775e+00,1.000000,-0.988548,0.001701,0.000000
25%,2202.750000,0.000000,0.000000,0.000000,105.489450,6.953115e+02,2202.750000,-0.158379,0.038533,0.000000
50%,4404.500000,0.000000,0.000000,0.000000,185.532630,1.854509e+03,4404.500000,-0.044169,0.094852,0.000000
75%,6606.250000,0.000000,0.000000,0.000000,424.417978,8.925927e+03,6606.250000,0.136595,0.149967,0.797033
max,8808.000000,191.000000,1362.000000,12950.073307,22017.296529,3.930649e+06,8808.000000,0.951699,0.613585,19.454017


## 8) Filter polygons by NDSI and NIR

Rule requested:
- remove polygons having **NDSI > 0.4** and **NIR > 0.15**

This means a polygon is removed only if **both** conditions are true.


In [12]:
filtered_1 = lakes_gee[
    ~(
        (lakes_gee["mean_ndsi"] > NDSI_THRESHOLD) &
        (lakes_gee["mean_nir"] > NIR_THRESHOLD)
    )
].copy()

print("After NDSI/NIR filtering:", len(filtered_1), "of", len(lakes_gee))


After NDSI/NIR filtering: 8659 of 8808


## 9) Remove polygons with no glacier within 10 km

In [13]:
filtered_2 = add_glacier_distance_km(filtered_1, glaciers)
filtered_2 = filtered_2[filtered_2["glacier_dist_km"] <= GLACIER_DISTANCE_KM].copy()

print("After glacier-distance filtering:", len(filtered_2))
print(filtered_2[["lake_id", "glacier_dist_km"]].head())


C:\Users\gg\AppData\Local\Temp\ipykernel_22236\3877100771.py:72: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = gdf.to_crs(4326).unary_union.centroid


After glacier-distance filtering: 8576
   lake_id  glacier_dist_km
0        1         0.000000
1        2         0.020325
2        3         0.000000
3        4         0.245418
4        5         0.880240


In [14]:
# Save back in the original lake CRS if currently projected
# filtered_2 was projected to a metric CRS in add_glacier_distance_km, so convert back to original lake CRS.
filtered_2_save = filtered_2.to_crs(lakes.crs)

filtered_2_save.to_file(OUTPUT_SHP)

print("Final shapefile saved successfully:")
print(OUTPUT_SHP)


C:\Users\gg\AppData\Local\Temp\ipykernel_22236\206610694.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  filtered_2_save.to_file(OUTPUT_SHP)
c:\Users\gg\anaconda3\envs\thesis\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'glacier_dist_km' to 'glacier_di'
  ogr_write(


Final shapefile saved successfully:
D:\Thesis\glacial_lake_detection_thesis\Inference\4 Inference\post_process_outputs\filtered_lakes.shp


## 10) Remove polygons with mean slope > 15°

In [15]:
filtered_3 = add_mean_slope_to_polygons(filtered_2, SLOPE_TIF)
final_gdf = filtered_3[filtered_3["mean_slope_deg"] <= SLOPE_THRESHOLD_DEG].copy()

print("After slope filtering:", len(final_gdf))
print(final_gdf[["lake_id", "mean_slope_deg"]].head())


After slope filtering: 9
     lake_id  mean_slope_deg
0          1       12.676623
640      641        9.373909
652      653        6.922373
943      944       11.531810
992      993       12.577140


## 11) Save final shapefile

In [ ]:
# Save back in the original lake CRS if currently projected
# filtered_2 was projected to a metric CRS in add_glacier_distance_km, so convert back to original lake CRS.
final_gdf = final_gdf.to_crs(lakes.crs)

final_gdf.to_file(OUTPUT_SHP)

print("Final shapefile saved successfully:")
print(OUTPUT_SHP)


## 12) Optional quick summary

In [ ]:
summary = {
    "input_lakes": len(lakes),
    "after_ndsi_nir_filter": len(filtered_1),
    "after_glacier_distance_filter": len(filtered_2),
    "final_polygons": len(final_gdf),
}
summary


## Troubleshooting

### Earth Engine authentication issues
- Make sure you have access to Google Earth Engine.
- Make sure `EE_PROJECT_ID` is set to a valid Google Cloud project.

### No imagery returned
- Change `START_DATE` and `END_DATE`.
- Expand the date range.
- Check whether your study area has sufficient Sentinel-2 coverage in that period.

### Missing CRS
- Ensure DEM, lake shapefile, and glacier shapefile all have proper CRS definitions.

### Slope statistic choice
This notebook uses **mean slope per polygon**.  
If your methodology requires **maximum slope**, change the zonal statistics from `stats=["mean"]` to `stats=["max"]` and filter on that field instead.
